# Phase 3b — Predicting recommendations with Spark MLlib

Phases 2 and 3a established *that* playtime tracks the recommendation rate.
This notebook turns the question into a supervised learning problem: given
what we know about a review and the game it belongs to, can we predict
whether the player recommended it — and which of those signals actually
carries the weight?

Requires HDFS to be running (`start-dfs.sh`, `start-yarn.sh`).

In [1]:
import getpass

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
USER = getpass.getuser()
HDFS_BASE = f"hdfs://localhost:9000/user/{USER}/steam"
INPUT_PATH = f"{HDFS_BASE}/streaming_input/*.csv"

spark = (
    SparkSession.builder
    .appName("steam-reviews-ml")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

26/09/08 21:33:01 WARN Utils: Your hostname, luca-Katana-15-B13VFK resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlo1)
26/09/08 21:33:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/08 21:33:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Features

Five predictors, deliberately few: the numeric ones are taken as they are,
and the price bucket is one-hot encoded. `is_recommended` becomes the label.

`positive_ratio` is the share of positive reviews the game has on the store,
so it describes the game's reputation rather than this particular player's
experience.

In [3]:
raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)

data = (
    raw
    .withColumn("year", F.year("date"))
    .withColumn("label", F.col("is_recommended").cast("int"))
    .select("hours", "price_final", "positive_ratio", "year", "price_bucket", "label")
    .dropna()
)

print(f"rows: {data.count():,}")
data.show(5)

rows: 500,000
+-----+-----------+--------------+----+------------+-----+
|hours|price_final|positive_ratio|year|price_bucket|label|
+-----+-----------+--------------+----+------------+-----+
|558.5|       20.0|            83|2020|         mid|    1|
| 52.7|       20.0|            89|2020|         mid|    1|
|823.5|       15.0|            88|2020|         mid|    1|
|614.7|       15.0|            88|2020|         mid|    1|
| 10.0|       20.0|            83|2020|         mid|    0|
+-----+-----------+--------------+----+------------+-----+
only showing top 5 rows



In [4]:
# Class balance: worth knowing before reading any accuracy figure
data.groupBy("label").count().withColumn(
    "share", F.round(F.col("count") / data.count(), 4)
).show()

+-----+------+------+
|label| count| share|
+-----+------+------+
|    1|422623|0.8452|
|    0| 77377|0.1548|
+-----+------+------+



In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

NUMERIC = ["hours", "price_final", "positive_ratio", "year"]

indexer = StringIndexer(inputCol="price_bucket", outputCol="price_idx")
encoder = OneHotEncoder(inputCols=["price_idx"], outputCols=["price_vec"])
assembler = VectorAssembler(inputCols=NUMERIC + ["price_vec"], outputCol="features")

prep = Pipeline(stages=[indexer, encoder, assembler]).fit(data)
prepared = prep.transform(data).select("features", "label").cache()

prepared.show(3, truncate=False)

[Stage 17:>                                                       (0 + 16) / 17]

+------------------------------------+-----+
|features                            |label|
+------------------------------------+-----+
|[558.5,20.0,83.0,2020.0,0.0,1.0,0.0]|1    |
|[52.7,20.0,89.0,2020.0,0.0,1.0,0.0] |1    |
|[823.5,15.0,88.0,2020.0,0.0,1.0,0.0]|1    |
+------------------------------------+-----+
only showing top 3 rows



## Train / test split

A fixed seed keeps the split reproducible across runs.

In [6]:
train, test = prepared.randomSplit([0.8, 0.2], seed=42)

print(f"train: {train.count():,}")
print(f"test:  {test.count():,}")

train: 400,336
test:  99,664


## Two models

Logistic regression as an interpretable baseline, random forest as a model
that can pick up non-linearities and interactions. Both are trained on the
same split so the comparison is fair.

In [7]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, seed=42)

lr_model = lr.fit(train)
rf_model = rf.fit(train)

print("both models trained")

both models trained


## Evaluation

AUC measures ranking quality and is unaffected by the class imbalance;
accuracy and F1 describe the hard predictions at the default threshold.

In [8]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")


def evaluate(name, model, test_df):
    preds = model.transform(test_df)
    return {
        "model": name,
        "auc": round(auc_eval.evaluate(preds), 4),
        "accuracy": round(acc_eval.evaluate(preds), 4),
        "f1": round(f1_eval.evaluate(preds), 4),
    }


results = [
    evaluate("logistic_regression", lr_model, test),
    evaluate("random_forest", rf_model, test),
]

for r in results:
    print(r)

{'model': 'logistic_regression', 'auc': 0.7202, 'accuracy': 0.8423, 'f1': 0.7895}
{'model': 'random_forest', 'auc': 0.7609, 'accuracy': 0.8525, 'f1': 0.8013}


In [9]:
# A majority-class baseline: always predicting "recommended".
# Any model has to beat this to be worth anything.
majority = test.filter(F.col("label") == 1).count() / test.count()
print(f"baseline accuracy (always predict 1): {majority:.4f}")

baseline accuracy (always predict 1): 0.8446


In [10]:
# Confusion matrix for the better-performing model
best_name = max(results, key=lambda r: r["auc"])["model"]
best_model = rf_model if best_name == "random_forest" else lr_model
print(f"confusion matrix — {best_name}")

(
    best_model.transform(test)
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
    .show()
)

confusion matrix — random_forest


[Stage 120:=========>                                             (3 + 14) / 17]

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1410|
|    0|       1.0|14076|
|    1|       0.0|  622|
|    1|       1.0|83556|
+-----+----------+-----+

